# NumPy 100 — Problems 51–60

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 51. Create a structured array representing a position (x,y) and a color (r,g,b) (★★☆)

In [2]:
Z = np.zeros(10, dtype=[("position", [("x", float),
                                      ("y", float)]),
                        ("color",    [("r", float),
                                      ("g", float),
                                      ("b", float)])])
print(Z.dtype)

# fill and read nested fields
Z["position"]["x"] = np.linspace(0, 1, 10)
Z["color"]["r"] = 1.0
print(Z[:3])
print(Z["position"]["x"])

[('position', [('x', '<f8'), ('y', '<f8')]), ('color', [('r', '<f8'), ('g', '<f8'), ('b', '<f8')])]
[((0.        , 0.), (1., 0., 0.)) ((0.11111111, 0.), (1., 0., 0.))
 ((0.22222222, 0.), (1., 0., 0.))]
[0.         0.11111111 0.22222222 0.33333333 0.44444444 0.55555556
 0.66666667 0.77777778 0.88888889 1.        ]


**Concept:** structured dtypes can be nested: a field's type can itself be a list of named fields. Access nested values by chaining field names, e.g. `Z["position"]["x"]`. Each element then behaves like a small record with a position and a color.

**Needed help?** solution  
**Revisit on:** Nov 12

---
### 52. Consider a random vector with shape (100,2) representing coordinates, find point by point distances (★★☆)

In [3]:
rng = np.random.default_rng(42)
Z = rng.random((100, 2))
X, Y = Z[:, 0], Z[:, 1]

# column vector minus row vector -> every pair, via broadcasting
D = np.sqrt((X[:, None] - X[None, :])**2 + (Y[:, None] - Y[None, :])**2)

print(D.shape)                                       # (100, 100)
print(np.allclose(np.diag(D), 0))                    # True: distance to itself is 0
print(np.allclose(D, D.T))                           # True: symmetric
print(np.isclose(D[3, 7], np.linalg.norm(Z[3] - Z[7])))   # True: spot-check one pair
print(D[:3, :3].round(3))

# alternative (needs SciPy): from scipy.spatial.distance import cdist; cdist(Z, Z)

(100, 100)
True
True
True
[[0.    0.272 0.866]
 [0.272 0.    0.813]
 [0.866 0.813 0.   ]]


**Concept:** broadcasting builds all pairwise differences without loops: a column `(100, 1)` minus a row `(1, 100)` gives a `(100, 100)` matrix. Apply the Euclidean formula `sqrt(dx² + dy²)` to get the distance matrix. It is symmetric with zeros on the diagonal. `scipy.spatial.distance.cdist` does the same job.

**Needed help?** solution  
**Revisit on:** Nov 12

---
### 53. How to convert a float (32 bits) array into an integer (32 bits) array in place?

In [4]:
rng = np.random.default_rng(42)
Z = (rng.random(10) * 100).astype(np.float32)
original = Z.copy()
print(original)

Y = Z.view(np.int32)                    # same memory, now interpreted as int32
Y[:] = Z                                # convert float values to ints, writing into that memory

print(Y)
print(np.shares_memory(Y, Z))           # True: no new buffer was allocated
print(np.array_equal(Y, original.astype(np.int32)))   # True: same result as a normal cast

[77.39561  43.887844 85.859795 69.7368    9.417735 97.56223  76.11397
 78.60643  12.811363 45.038593]
[77 43 85 69  9 97 76 78 12 45]
True
True


**Concept:** `Z.view(np.int32)` reinterprets the same memory as 32-bit integers (float32 and int32 are both 4 bytes). Writing `Y[:] = Z` then stores the *converted* (truncated) values into that same buffer, so the conversion is done in place. After this, reading `Z` as floats gives meaningless numbers because the bytes now hold ints. A plain `Z.astype(np.int32)` would allocate a new array.

**Needed help?** solution  
**Revisit on:** Nov 12

---
### 54. How to read the following file? (★★☆)
```
1, 2, 3, 4, 5
6,  ,  , 7, 8
 ,  , 9,10,11
```

In [5]:
from io import StringIO

s = StringIO('''1, 2, 3, 4, 5
6,  ,  , 7, 8
 ,  , 9,10,11''')

Z = np.genfromtxt(s, delimiter=",")
print(Z)                                # missing values become nan

# integer version: missing values become -1
s.seek(0)
print(np.genfromtxt(s, delimiter=",", dtype=int))

# for a real file on disk: np.genfromtxt("data.csv", delimiter=",")

[[ 1.  2.  3.  4.  5.]
 [ 6. nan nan  7.  8.]
 [nan nan  9. 10. 11.]]
[[ 1  2  3  4  5]
 [ 6 -1 -1  7  8]
 [-1 -1  9 10 11]]


**Concept:** `np.genfromtxt(file, delimiter=",")` reads text data and handles missing entries, which `np.loadtxt` cannot. Blanks become `nan` for float output, or `-1` when `dtype=int`. `StringIO` lets you treat a string like a file for testing. Use `filling_values=` to choose your own replacement.

**Needed help?** solution  
**Revisit on:** Nov 12

---
### 55. What is the equivalent of enumerate for numpy arrays? (★★☆)

In [6]:
Z = np.arange(9).reshape(3, 3)

# enumerate equivalent: yields ((row, col), value)
for index, value in np.ndenumerate(Z):
    print(index, value)

# only the indices:
print(list(np.ndindex(Z.shape)))

(0, 0) 0
(0, 1) 1
(0, 2) 2
(1, 0) 3
(1, 1) 4
(1, 2) 5
(2, 0) 6
(2, 1) 7
(2, 2) 8
[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]


**Concept:** `np.ndenumerate(Z)` is the NumPy version of `enumerate`: it yields `(index_tuple, value)` for every element of an array of any number of dimensions. `np.ndindex(shape)` yields only the index tuples. Loops over arrays are slow, so use these only when vectorization isn't possible.

**Needed help?** hint  
**Revisit on:** Nov 12

---
### 56. Generate a generic 2D Gaussian-like array (★★☆)

In [7]:
X, Y = np.meshgrid(np.linspace(-1, 1, 11), np.linspace(-1, 1, 11))
D = np.sqrt(X * X + Y * Y)              # distance from the center
sigma, mu = 1.0, 0.0

G = np.exp(-((D - mu) ** 2 / (2.0 * sigma ** 2)))

with np.printoptions(precision=2, suppress=True):
    print(G)
print(G[5, 5], G.max())                 # 1.0 at the center

[[0.37 0.44 0.51 0.56 0.59 0.61 0.59 0.56 0.51 0.44 0.37]
 [0.44 0.53 0.61 0.67 0.71 0.73 0.71 0.67 0.61 0.53 0.44]
 [0.51 0.61 0.7  0.77 0.82 0.84 0.82 0.77 0.7  0.61 0.51]
 [0.56 0.67 0.77 0.85 0.9  0.92 0.9  0.85 0.77 0.67 0.56]
 [0.59 0.71 0.82 0.9  0.96 0.98 0.96 0.9  0.82 0.71 0.59]
 [0.61 0.73 0.84 0.92 0.98 1.   0.98 0.92 0.84 0.73 0.61]
 [0.59 0.71 0.82 0.9  0.96 0.98 0.96 0.9  0.82 0.71 0.59]
 [0.56 0.67 0.77 0.85 0.9  0.92 0.9  0.85 0.77 0.67 0.56]
 [0.51 0.61 0.7  0.77 0.82 0.84 0.82 0.77 0.7  0.61 0.51]
 [0.44 0.53 0.61 0.67 0.71 0.73 0.71 0.67 0.61 0.53 0.44]
 [0.37 0.44 0.51 0.56 0.59 0.61 0.59 0.56 0.51 0.44 0.37]]
1.0 1.0


**Concept:** build 2D coordinate grids with `np.meshgrid`, compute each point's distance from the center, then apply the Gaussian formula `exp(-(d - mu)² / (2σ²))` to the whole array at once. The peak is 1 at the center and falls off with distance; `sigma` controls the width.

**Needed help?** solution  
**Revisit on:** Nov 12

---
### 57. How to randomly place p elements in a 2D array? (★★☆)

In [8]:
rng = np.random.default_rng(42)
n = 10
p = 3

Z = np.zeros((n, n))
positions = rng.choice(n * n, p, replace=False)     # p unique flat positions
np.put(Z, positions, 1)

print(Z)
print(Z.sum())                          # 3.0: exactly p elements were placed

[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
3.0


**Concept:** pick `p` *unique* positions with `rng.choice(n*n, p, replace=False)` (without `replace=False` the same cell could be chosen twice). `np.put(Z, indices, value)` writes into the array using flat (1D) indices, so you don't need to convert to row and column. `Z.flat[positions] = 1` does the same.

**Needed help?** hint  
**Revisit on:** Nov 12

---
### 58. Subtract the mean of each row of a matrix (★★☆)

In [9]:
rng = np.random.default_rng(42)
X = rng.random((3, 4))

Y = X - X.mean(axis=1, keepdims=True)   # keepdims keeps the shape (3, 1) so it broadcasts

print(X)
print(Y)
print(np.allclose(Y.mean(axis=1), 0))   # True: every row now has mean 0

[[0.77395605 0.43887844 0.85859792 0.69736803]
 [0.09417735 0.97562235 0.7611397  0.78606431]
 [0.12811363 0.45038594 0.37079802 0.92676499]]
[[ 0.08175594 -0.25332167  0.16639781  0.00516792]
 [-0.56007358  0.32137142  0.10688878  0.13181338]
 [-0.34090201 -0.01862971 -0.09821762  0.45774934]]
True


**Concept:** `X.mean(axis=1)` averages across each row. Without `keepdims=True` the result has shape `(3,)`, which lines up with the *columns*: it raises an error here (3 vs 4) and silently gives wrong numbers for a square matrix. `keepdims=True` keeps shape `(3, 1)` so it subtracts from each row correctly. Older code uses `.reshape(-1, 1)` for the same effect.

**Needed help?** hint  
**Revisit on:** Nov 12

---
### 59. How to sort an array by the nth column? (★★☆)

In [10]:
rng = np.random.default_rng(42)
Z = rng.integers(0, 10, (3, 3))
print(Z)

n = 1                                   # sort by column 1
print(Z[Z[:, n].argsort()])

[[0 7 6]
 [4 4 8]
 [0 6 2]]
[[4 4 8]
 [0 6 2]
 [0 7 6]]


**Concept:** `Z[:, n].argsort()` gives the row order that would sort column `n`, and indexing `Z[order]` reorders whole rows by that order. This keeps each row intact, unlike `Z[:, n].sort()`, which would sort only one column.

**Needed help?** hint  
**Revisit on:** Nov 12

---
### 60. How to tell if a given 2D array has null columns? (★★☆)

In [11]:
Z = np.array([[1, 0, 3, 0],
              [4, 0, 6, 7]])            # column 1 is all zeros

print((~Z.any(axis=0)).any())           # True: there is a null column
print(np.where(~Z.any(axis=0))[0])      # [1]: which column(s)

Z2 = np.array([[1, 2],
               [3, 4]])
print((~Z2.any(axis=0)).any())          # False

True
[1]
False


**Concept:** `Z.any(axis=0)` is True for each column that has at least one non-zero value. Negating it with `~` marks the *null* (all-zero) columns, and `.any()` collapses that to one True/False. `np.where(...)` tells you which columns. Use `axis=1` for rows instead.

**Needed help?** hint  
**Revisit on:** Nov 12